# Gaussian Blur Sağlamlık Deneyi — Ahmet

**Tek değişken: sigma** (0 = filtresiz referans, 0.5, 1.5, 3.0; çekirdek 15×15). Çözünürlük, baş ROI, renk aralıkları ve karar eşikleri sabit.
Her K görüntüsü ve sigma için RGB/HSV maske alanı, baş örtüşmesi, sınıf ve TP/FP/FN ölçülür. Bulanık kopyalar yeni bağımsız görüntü sayılmaz.

In [ ]:
from pathlib import Path
import sys
KOK = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kaynak" / "analiz.py").is_file())
sys.path.insert(0, str(KOK))

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from kaynak.veri import ayarlari_oku, veri_setini_oku, goruntu_oku, maske_oku
from kaynak.analiz import analiz_et
from kaynak.gorsel import bgr2rgb, kutulari_ciz, maske_bindir
ayarlar = ayarlari_oku()
kayitlar = veri_setini_oku()
pd.set_option("display.max_colwidth", 80)
print("Görüntüler:", ", ".join(f"{k['kimlik']} ({k['bolum']})" for k in kayitlar))

## 1. Deney öncesi tahmin

In [ ]:
tahminler = pd.read_csv(KOK / "belgeler/tahminler.csv")
benim = tahminler[(tahminler.uye == "Ahmet") & tahminler.deney_kimligi.str.startswith("gaussian")]
if benim.empty:
    print("UYARI: Bu deney için kayıtlı tahmin yok. Hoca: 'Tahmini olmayan deney puanlanmaz.'")
    print("Deneyi çalıştırmadan ÖNCE belgeler/tahminler.csv dosyasına iki cümle ekleyin.")
for _, t in benim.iterrows():
    print(f"[{t.kayit_tarihi}] {t.deney_kimligi}\n 1) {t.tahmin_1}\n 2) {t.tahmin_2}")

## 2. Sonuçlar (`python betikler/deneyleri_calistir.py` çıktısı)

In [ ]:
b = pd.read_csv(KOK / "sonuclar/bulaniklik/bulaniklik_sonuclari.csv")
b[["kimlik", "sigma", "yontem", "maske_alani_yuzde", "bas_ortusmesi_yuzde", "kural_sinifi", "dogru_sinif", "dogru_mu", "tp", "fp", "fn"]]

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(KOK / "sonuclar/bulaniklik/sigma_maske_alani.png")))

In [ ]:
for k in ("K1", "K2", "K3", "K4"):
    display(Image(filename=str(KOK / f"sonuclar/bulaniklik/{k}_sigma.png"), width=900))

## 3. Sınıf değişimi

In [ ]:
import json
print(json.loads((KOK / "sonuclar/ozet.json").read_text(encoding="utf-8"))["bulaniklik_sinif_degisimi"] or "Hiçbir görüntüde sınıf değişmedi.")

## 4. Yorum
*(Üye yazar: küçük baret örneğinde ayrıntı kaybı, aynı renkli nesnede yanlış alarm; maskenin artması veya azalması önceden garanti değildir.)*